# 🚀 DiabetAI v10 - Ghost Protocol Edition

**Critical Fix:** `is_synthetic` used ONLY for weighting, then DROPPED

**Strategy:**
- 2M synthetic correction rows (CGM 140-450, carbs=0)
- Weight 2.0x on synthetic + real high BG
- `is_synthetic` never reaches the model
- v9 physics logic fully maintained

In [ ]:
!pip install polars xgboost matplotlib scikit-learn -U -q

import polars as pl
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import json
import os
import gc

from sklearn.metrics import mean_squared_error, mean_absolute_error

from google.colab import drive
drive.mount('/content/drive')

print(f"✅ XGBoost: {xgb.__version__}")
print(f"✅ Polars: {pl.__version__}")

# GPU Check
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], 
                       capture_output=True, text=True)
print(f"\n🖥️ GPU:\n{result.stdout}")

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"
MODEL_PATH = "/content/drive/MyDrive/DiabetesKDS/models"

TRAIN_FILE = f"{BASE_PATH}/processed/train_ready_balanced_v7.parquet"
TEST_FILE = f"{BASE_PATH}/processed/test_ready_real_world_v7.parquet"

TARGET = 'bolus'
TARGET_BG = 100.0

SYNTHETIC_SIZE = 2_000_000

print(f"📂 Train: {TRAIN_FILE}")
print(f"📂 Test: {TEST_FILE}")
print(f"\n🧬 Synthetic Injection: {SYNTHETIC_SIZE:,} rows")

In [ ]:
# ============================================================
# v10 FEATURE LIST (20 features - NO is_synthetic!)
# ============================================================
V10_FEATURES = [
    'CGM', 'carbs', 'iob',
    'phys_meal_bolus', 'phys_corr_bolus', 'phys_net_need',
    'est_tdd', 'est_isf', 'est_icr',
    'cgm_delta_5min', 'cgm_delta_15min', 'cgm_velocity',
    'basal', 'basal_delta_5min',
    'time_sin', 'time_cos', 'is_weekend',
    'age', 'weight', 'gender_encoded'
]

print(f"📋 v10 Features: {len(V10_FEATURES)}")
print(f"⚠️ is_synthetic is NOT in feature list (Ghost Protocol)")

## Step 1: Physics Feature Engineering (v9 Logic)

In [ ]:
def add_physics_features(df: pl.DataFrame) -> pl.DataFrame:
    """
    v9 Physics Logic - Applied to BOTH real and synthetic data.
    """
    df = df.with_columns([
        # Meal Bolus: carbs / ICR
        (pl.col('carbs') / pl.col('est_icr').clip(lower_bound=1.0))
            .fill_nan(0.0).fill_null(0.0)
            .alias('phys_meal_bolus'),
        
        # Correction Bolus: (CGM - target) / ISF
        ((pl.col('CGM') - TARGET_BG) / pl.col('est_isf').clip(lower_bound=1.0))
            .clip(lower_bound=0.0)
            .fill_nan(0.0).fill_null(0.0)
            .alias('phys_corr_bolus'),
    ])
    
    df = df.with_columns([
        (pl.col('phys_meal_bolus') + pl.col('phys_corr_bolus') - pl.col('iob'))
            .clip(lower_bound=0.0)
            .alias('phys_net_need')
    ])
    
    return df

print("✅ add_physics_features() defined (v9 logic)")

## Step 2: Generate 2M Synthetic Correction Rows

In [ ]:
print("="*70)
print("🧬 GENERATING SYNTHETIC DATA")
print("="*70)

np.random.seed(42)

def generate_synthetic_corrections(n=2_000_000):
    """Generate synthetic HIGH BG correction examples."""
    print(f"\n🔄 Generating {n:,} synthetic correction rows...")
    
    # CGM: Focus on HIGH (140-450, biased towards very high)
    cgm_low = np.random.randint(140, 200, size=n // 3)
    cgm_mid = np.random.randint(200, 300, size=n // 3)
    cgm_high = np.random.randint(300, 451, size=n - 2*(n//3))
    cgm = np.concatenate([cgm_low, cgm_mid, cgm_high]).astype(np.float32)
    np.random.shuffle(cgm)
    
    # Carbs: STRICTLY 0
    carbs = np.zeros(n, dtype=np.float32)
    
    # ISF: Cover all sensitivities
    isf_options = np.array([20, 25, 30, 35, 40, 45, 50, 60], dtype=np.float32)
    est_isf = np.random.choice(isf_options, size=n)
    
    # ICR
    icr_options = np.array([8, 10, 12, 15, 18, 20], dtype=np.float32)
    est_icr = np.random.choice(icr_options, size=n)
    
    # IOB: 0 to 4 units
    iob = np.random.uniform(0.0, 4.0, size=n).astype(np.float32)
    
    # TDD
    est_tdd = (1800.0 / est_isf).astype(np.float32)
    
    # Time features
    hours = np.random.uniform(0, 24, size=n)
    time_sin = np.sin(2 * np.pi * hours / 24).astype(np.float32)
    time_cos = np.cos(2 * np.pi * hours / 24).astype(np.float32)
    is_weekend = np.random.choice([0.0, 1.0], size=n, p=[0.7, 0.3]).astype(np.float32)
    
    # Trends
    cgm_delta_5min = np.random.uniform(-2, 5, size=n).astype(np.float32)
    cgm_delta_15min = (cgm_delta_5min * 3).astype(np.float32)
    cgm_velocity = (cgm_delta_15min / 15).astype(np.float32)
    
    # Basal
    basal = np.random.uniform(0.5, 2.0, size=n).astype(np.float32)
    basal_delta_5min = np.zeros(n, dtype=np.float32)
    
    # Demographics
    age = np.random.randint(18, 70, size=n).astype(np.float32)
    weight = np.random.randint(50, 120, size=n).astype(np.float32)
    gender_encoded = np.random.choice([0.0, 1.0], size=n).astype(np.float32)
    
    # THE GOLDEN LABEL: (CGM - 100) / ISF - IOB
    bolus = np.maximum(0, (cgm - TARGET_BG) / est_isf - iob).astype(np.float32)
    
    df = pl.DataFrame({
        'CGM': cgm,
        'carbs': carbs,
        'iob': iob,
        'est_tdd': est_tdd,
        'est_isf': est_isf,
        'est_icr': est_icr,
        'cgm_delta_5min': cgm_delta_5min,
        'cgm_delta_15min': cgm_delta_15min,
        'cgm_velocity': cgm_velocity,
        'basal': basal,
        'basal_delta_5min': basal_delta_5min,
        'time_sin': time_sin,
        'time_cos': time_cos,
        'is_weekend': is_weekend,
        'age': age,
        'weight': weight,
        'gender_encoded': gender_encoded,
        'bolus': bolus,
        'is_synthetic': np.ones(n, dtype=np.float32)  # TEMPORARY - will be dropped!
    })
    
    return df

df_synthetic = generate_synthetic_corrections(n=SYNTHETIC_SIZE)

# Apply physics features
df_synthetic = add_physics_features(df_synthetic)

print(f"\n✅ Synthetic data: {len(df_synthetic):,} rows")
print(f"📊 Bolus range: {df_synthetic.select('bolus').min()[0,0]:.2f} - {df_synthetic.select('bolus').max()[0,0]:.2f}")

## Step 3: Load Real Data

In [ ]:
print("="*70)
print("📂 LOADING REAL DATA")
print("="*70)

df_real = pl.read_parquet(TRAIN_FILE)
print(f"✅ Real data: {len(df_real):,} rows")

# Apply physics features
df_real = add_physics_features(df_real)

# Add is_synthetic=0 tag (TEMPORARY)
df_real = df_real.with_columns(pl.lit(0.0).alias('is_synthetic'))

print("✅ Physics features added")
print("✅ is_synthetic=0 tag added (temporary)")

## Step 4: Merge & Ghost Protocol Weighting

In [ ]:
print("="*70)
print("🔗 MERGE & GHOST PROTOCOL")
print("="*70)

# Align columns
weight_cols = V10_FEATURES + ['bolus', 'is_synthetic']

# Select only needed columns
df_real_aligned = df_real.select(weight_cols)
df_synthetic_aligned = df_synthetic.select(weight_cols)

# Merge
df_combined = pl.concat([df_real_aligned, df_synthetic_aligned])
df_combined = df_combined.sample(fraction=1.0, shuffle=True, seed=42)

print(f"\n✅ Combined: {len(df_combined):,} rows")

# Memory cleanup
del df_real, df_real_aligned, df_synthetic, df_synthetic_aligned
gc.collect()

In [ ]:
# ============================================================
# THE GHOST PROTOCOL: Calculate weights, then DROP is_synthetic
# ============================================================
print("\n👻 GHOST PROTOCOL: Calculating weights...")

# Extract for weight calculation
is_synthetic = df_combined.select('is_synthetic').to_numpy().ravel()
cgm_values = df_combined.select('CGM').to_numpy().ravel()

# Weight Strategy:
# - Synthetic data: Weight 2.0 (high priority)
# - Real + High BG (>180): Weight 2.0 (also high priority)
# - Real + Normal: Weight 1.0

sample_weights = np.ones(len(df_combined), dtype=np.float32)

# Synthetic = 2.0
sample_weights[is_synthetic == 1.0] = 2.0

# Real + High BG = 2.0
real_high_bg = (is_synthetic == 0.0) & (cgm_values > 180)
sample_weights[real_high_bg] = 2.0

print(f"\n📊 Weight Distribution:")
print(f"   Synthetic (2.0): {(is_synthetic == 1.0).sum():,}")
print(f"   Real High BG (2.0): {real_high_bg.sum():,}")
print(f"   Real Normal (1.0): {((is_synthetic == 0.0) & (cgm_values <= 180)).sum():,}")
print(f"   Total Effective Weight: {sample_weights.sum():,.0f}")

In [ ]:
# ============================================================
# 🚨 CRITICAL: DROP is_synthetic BEFORE creating X_train!
# ============================================================
print("\n🗑️ DROPPING is_synthetic from dataset...")

# Extract features (WITHOUT is_synthetic)
X_train = df_combined.select(V10_FEATURES).to_numpy().astype(np.float32)
y_train = df_combined.select('bolus').to_numpy().ravel().astype(np.float32)

# Clean inf/nan
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
y_train = np.nan_to_num(y_train, nan=0.0, posinf=0.0, neginf=0.0)

print(f"\n✅ X_train shape: {X_train.shape}")
print(f"✅ Features: {len(V10_FEATURES)} (is_synthetic NOT included!)")

# Verify is_synthetic is NOT in features
assert 'is_synthetic' not in V10_FEATURES, "ERROR: is_synthetic in feature list!"
print("\n👻 Ghost Protocol verified: is_synthetic eliminated!")

# Memory cleanup
del df_combined, is_synthetic, cgm_values, real_high_bg
gc.collect()

In [ ]:
# Load test data
print("\n📂 Loading test data...")
df_test = pl.read_parquet(TEST_FILE)
df_test = add_physics_features(df_test)

X_test = df_test.select(V10_FEATURES).to_numpy().astype(np.float32)
y_test = df_test.select('bolus').to_numpy().ravel().astype(np.float32)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

print(f"✅ Test: {X_test.shape}")

## Step 5: Monotonic Constraints (v9 Inherited)

In [ ]:
print("="*70)
print("🔒 MONOTONIC CONSTRAINTS (v9 Inheritance)")
print("="*70)

CONSTRAINTS = {
    'CGM': 1,               # Higher glucose -> more dose
    'carbs': 1,             # More carbs -> more dose
    'iob': -1,              # More IOB -> less dose
    'phys_meal_bolus': 1,   # Pre-calculated meal need
    'phys_corr_bolus': 1,   # Pre-calculated correction
    'phys_net_need': 1,     # Net insulin need
    'est_isf': -1,          # Higher sensitivity -> less per unit
    'est_icr': -1,          # Higher ratio -> less per carb
    'cgm_delta_5min': 1,    # Rising trend -> more
    'cgm_delta_15min': 1,
    'cgm_velocity': 1,
}

monotone_tuple = tuple(CONSTRAINTS.get(f, 0) for f in V10_FEATURES)

print("\n📋 Active Constraints:")
for i, (f, c) in enumerate(zip(V10_FEATURES, monotone_tuple)):
    if c != 0:
        symbol = "⬆️ +1" if c == 1 else "⬇️ -1"
        print(f"   {i:2d}: {f:<20} {symbol}")

constrained = sum(1 for c in monotone_tuple if c != 0)
print(f"\n✅ Constrained: {constrained}/{len(V10_FEATURES)}")

## Step 6: Train XGBoost v10

In [ ]:
print("="*70)
print("🚀 TRAINING XGBOOST v10")
print("="*70)

params = {
    'n_estimators': 3000,
    'learning_rate': 0.02,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'objective': 'reg:squarederror',
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': 42,
    'monotone_constraints': monotone_tuple,
    'early_stopping_rounds': 100
}

print(f"\n📊 Training: {len(X_train):,} rows")
print(f"📊 Effective: {sample_weights.sum():,.0f} (weighted)")

In [ ]:
print("\n🏋️ Training...")

model = xgb.XGBRegressor(**params)

model.fit(
    X_train, y_train,
    sample_weight=sample_weights,
    eval_set=[(X_test, y_test)],
    verbose=200
)

print(f"\n✅ Training Complete!")
print(f"   Best Iteration: {model.best_iteration}")
print(f"   Best Score: {model.best_score:.6f}")

## Step 7: THE FINAL VERDICT

In [ ]:
print("="*70)
print("🎯 THE FINAL VERDICT - 4-SCENARIO STRESS TEST")
print("="*70)

def create_test_input(cgm, carbs, iob, est_isf=30, est_icr=10, velocity=0):
    phys_meal = carbs / est_icr if est_icr > 0 else 0
    phys_corr = max(0, (cgm - TARGET_BG) / est_isf) if est_isf > 0 else 0
    phys_net = max(0, phys_meal + phys_corr - iob)
    
    return np.array([[
        cgm, carbs, iob,
        phys_meal, phys_corr, phys_net,
        40.0, est_isf, est_icr,
        velocity, velocity*3, velocity/5,
        1.0, 0.0,
        0.0, 1.0, 0.0,
        30.0, 70.0, 0.0
    ]], dtype=np.float32)

def calc_math_dose(cgm, carbs, iob, isf=30, icr=10, target=100):
    meal = carbs / icr if icr > 0 and carbs > 0 else 0
    corr = max(0, (cgm - target) / isf) if cgm > target else 0
    return max(0, meal + corr - iob)

scenarios = [
    {"name": "A: Meal (120, 50g)", "cgm": 120, "carbs": 50, "iob": 0, "velocity": 0,
     "target_range": (4.5, 7.5)},
    {"name": "B: Correction (250, 0g)", "cgm": 250, "carbs": 0, "iob": 0, "velocity": 2,
     "target_range": (4.5, 6.5)},  # CRITICAL: Must be > 4.5U!
    {"name": "C: Meal+Correction", "cgm": 180, "carbs": 40, "iob": 1, "velocity": 1,
     "target_range": (4.0, 8.0)},
    {"name": "D: Hypo Risk (80)", "cgm": 80, "carbs": 0, "iob": 0, "velocity": -3,
     "target_range": (0.0, 0.5)},  # CRITICAL: Must be < 0.5U!
]

print(f"\n{'Scenario':<25} {'Math':>8} {'v10':>8} {'Target':>12} {'Status':>8}")
print("-"*65)

all_pass = True
results = []

for s in scenarios:
    X = create_test_input(s['cgm'], s['carbs'], s['iob'], velocity=s['velocity'])
    math_dose = calc_math_dose(s['cgm'], s['carbs'], s['iob'])
    ai_dose = max(0, model.predict(X)[0])
    
    low, high = s['target_range']
    in_range = low <= ai_dose <= high
    status = "✅ PASS" if in_range else "❌ FAIL"
    
    if not in_range:
        all_pass = False
    
    results.append({'name': s['name'], 'math': math_dose, 'ai': ai_dose, 'status': status})
    
    target_str = f"{low:.1f}-{high:.1f}"
    print(f"{s['name']:<25} {math_dose:>8.2f} {ai_dose:>8.2f} {target_str:>12} {status:>8}")

print("-"*65)
if all_pass:
    print("\n🎉 ALL SCENARIOS PASSED!")
    print("✅ Correction Bolus Blindness: CURED!")
else:
    print("\n⚠️ Some scenarios need attention")

## Step 8: Save Model

In [ ]:
print("="*70)
print("💾 SAVING v10 MODEL")
print("="*70)

os.makedirs(MODEL_PATH, exist_ok=True)

model_file = f"{MODEL_PATH}/xgboost_clinical_v10.json"
model.save_model(model_file)
print(f"\n✅ Model: {model_file}")

config = {
    'version': 'v10',
    'features': V10_FEATURES,
    'feature_count': len(V10_FEATURES),
    'physics_features': ['phys_meal_bolus', 'phys_corr_bolus', 'phys_net_need'],
    'monotone_constraints': list(monotone_tuple),
    'target_bg': TARGET_BG,
    'synthetic_injection': SYNTHETIC_SIZE,
    'ghost_protocol': 'is_synthetic used for weighting only, then dropped',
    'best_iteration': model.best_iteration,
    'best_score': float(model.best_score),
    'stress_test': results
}

config_file = f"{MODEL_PATH}/model_config_v10.json"
with open(config_file, 'w') as f:
    json.dump(config, f, indent=2, default=str)
print(f"✅ Config: {config_file}")

print(f"\n🚀 v10 Ready for deployment!")
print(f"   Features: {len(V10_FEATURES)}")
print(f"   Synthetic: {SYNTHETIC_SIZE:,}")
print(f"   Ghost Protocol: ✅")